In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
import matplotlib.pyplot as plt

In [3]:
# Step 1: Load the dataset
data = pd.read_csv(r'C:\Users\Faculty\Desktop\Manoj_Honors\RF_csv\all_available(t-1)_with_spatial.csv')

# Step 2: Convert 'Time' to datetime format
data['Time'] = pd.to_datetime(data['Time'])

# Step 3: Set the Time column as the index
data.set_index('Time', inplace=True)

# Step 4: Select features and target variable
# Including PM2.5, NO2, Lat, and Lon as features
features = data[['PM2.5', 'Ozone', 'Lat', 'Lon','Spatial_Avg_PM2.5','Spatial_Avg_Ozone']]
target = data['NO2']

# Step 5: Normalize the features and target
scaler_features = MinMaxScaler(feature_range=(0, 1))
scaled_features = scaler_features.fit_transform(features)

scaler_target = MinMaxScaler(feature_range=(0, 1))
scaled_target = scaler_target.fit_transform(target.values.reshape(-1, 1))

# Step 6: Create sequences for LSTM
sequence_length = 24  # Using the last 24 time steps for prediction

X, y = [], []
for i in range(sequence_length, len(scaled_features)):
    X.append(scaled_features[i-sequence_length:i])  # Previous 24 intervals of features
    y.append(scaled_target[i])                       # Ozone value to predict

X, y = np.array(X), np.array(y)

# Reshape X to be [samples, time steps, features]
X = X.reshape((X.shape[0], X.shape[1], X.shape[2]))  # Shape: (samples, 24, 4)

# Step 7: Build the LSTM model
model = Sequential()
model.add(LSTM(units=50, return_sequences=True, input_shape=(X.shape[1], X.shape[2])))
model.add(Dropout(0.2))  # Optional dropout to reduce overfitting
model.add(LSTM(units=50, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(units=1))  # Output layer

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Step 8: Train the model
model.fit(X, y, epochs=50, batch_size=32)

# Step 9: Make predictions
predicted_ozone = model.predict(X)

# Rescale predictions back to original range
predicted_ozone = scaler_target.inverse_transform(predicted_ozone)
actual_ozone = scaler_target.inverse_transform(y.reshape(-1, 1))

# Step 10: Calculate evaluation metrics
mae = mean_absolute_error(actual_ozone, predicted_ozone)
mse = mean_squared_error(actual_ozone, predicted_ozone)
r2 = r2_score(actual_ozone, predicted_ozone)

print(f'Mean Absolute Error (MAE): {mae}')
print(f'Mean Squared Error (MSE): {mse}')
print(f'R-squared (R²): {r2}')

# Step 11: Plot actual vs predicted Ozone levels
plt.figure(figsize=(14, 5))
plt.plot(data.index[sequence_length:], actual_ozone, label='Actual Ozone', color='blue')
plt.plot(data.index[sequence_length:], predicted_ozone, label='Predicted Ozone', color='red')
plt.title('Ozone Prediction using LSTM')
plt.xlabel('Time')
plt.ylabel('Ozone Levels')
plt.legend()
plt.show()


c:\Users\Faculty\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
26767/26767 ━━━━━━━━━━━━━━━━━━━━ 155s 6ms/step - loss: 0.0043
Epoch 2/50
 1024/26767 ━━━━━━━━━━━━━━━━━━━━ 2:29 6ms/step - loss: 0.0034

KeyboardInterrupt: 

In [1]:
X = X.reshape((X.shape[0], X.shape[1], X.shape[2]))

NameError: name 'X' is not defined

In [8]:
# Load the dataset
all_available = pd.read_csv(r'C:\Users\Faculty\Desktop\Manoj_Honors\RF_csv\all_available(t-1)_with_spatial.csv')

# Prepare the feature set and target variable
X = all_available[['Time', 'Lat','Lon','PM2.5','NO2','Average_PM2.5_t-1' , 'Average_NO2_t-1']]
y = all_available['Ozone']

# Convert 'Time' to numeric if it's in datetime format
X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds

# Initialize the Random Forest model
rf = RandomForestRegressor(n_estimators=100, random_state=12)

# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=12)

# Prepare lists to store scores for each fold
mse_scores = []
mae_scores = []
r2_scores = []
mape_scores = []
nrmse_scores = []

# Perform cross-validation
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Fit the model on the training data
    rf.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred = rf.predict(X_test)

    # Calculate evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    nrmse = np.sqrt(mse) / (y_test.max() - y_test.min())

    mse_scores.append(mse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    mape_scores.append(mape)
    nrmse_scores.append(nrmse)

# Calculate mean and standard deviation for each metric
mean_mse = np.mean(mse_scores)
std_mse = np.std(mse_scores)
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)
mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)
mean_mape = np.mean(mape_scores)
std_mape = np.std(mape_scores)
mean_nrmse = np.mean(nrmse_scores)
std_nrmse = np.std(nrmse_scores)

# Print evaluation metrics
print(f"Mean Squared Error: {round(mean_mse, 2)} ± {round(std_mse, 2)}")
print(f"Mean Absolute Error: {round(mean_mae, 2)} ± {round(std_mae, 2)}")
print(f"R^2 Score: {round(mean_r2, 2)} ± {round(std_r2, 2)}")
print(f"Mean Absolute Percentage Error: {round(mean_mape, 2)}% ± {round(std_mape, 2)}%")
print(f"Normalized RMSE: {round(mean_nrmse, 4)} ± {round(std_nrmse, 4)}")


C:\Users\Faculty\AppData\Local\Temp\ipykernel_9408\593513356.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds


Mean Squared Error: 225.75 ± 1.27
Mean Absolute Error: 8.45 ± 0.03
R^2 Score: 0.8 ± 0.0
Mean Absolute Percentage Error: 104.58% ± 1.83%
Normalized RMSE: 0.0751 ± 0.0002


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the dataset
all_available = pd.read_csv(r'C:\Users\Faculty\Desktop\Manoj_Honors\RF_csv\all_available(t-1)_with_spatial.csv')

# Prepare the feature set with NO2 and the target variables
X = all_available[['Time', 'Lat', 'Lon', 'NO2', 'Spatial_Avg_NO2', 'Average_NO2_t-1']]
y = all_available[['PM2.5', 'Ozone']]  # Target variables: PM2.5 and Ozone

# Convert 'Time' to numeric if it's in datetime format
X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds

# Initialize the Random Forest model wrapped in MultiOutputRegressor
rf = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=12))

# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=12)

# Prepare lists to store scores for each fold for both targets
mse_scores_pm25 = []
mae_scores_pm25 = []
r2_scores_pm25 = []
mape_scores_pm25 = []
nrmse_scores_pm25 = []

mse_scores_ozone = []
mae_scores_ozone = []
r2_scores_ozone = []
mape_scores_ozone = []
nrmse_scores_ozone = []

# Perform cross-validation
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Fit the model on the training data
    rf.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred = rf.predict(X_test)

    # Calculate evaluation metrics for PM2.5
    mse_pm25 = mean_squared_error(y_test['PM2.5'], y_pred[:, 0])
    mae_pm25 = mean_absolute_error(y_test['PM2.5'], y_pred[:, 0])
    r2_pm25 = r2_score(y_test['PM2.5'], y_pred[:, 0])
    mape_pm25 = np.mean(np.abs((y_test['PM2.5'] - y_pred[:, 0]) / y_test['PM2.5'].replace(0, np.nan))) * 100  # Avoid division by zero
    nrmse_pm25 = np.sqrt(mse_pm25) / (y_test['PM2.5'].max() - y_test['PM2.5'].min())

    # Calculate evaluation metrics for Ozone
    mse_ozone = mean_squared_error(y_test['Ozone'], y_pred[:, 1])
    mae_ozone = mean_absolute_error(y_test['Ozone'], y_pred[:, 1])
    r2_ozone = r2_score(y_test['Ozone'], y_pred[:, 1])
    mape_ozone = np.mean(np.abs((y_test['Ozone'] - y_pred[:, 1]) / y_test['Ozone'].replace(0, np.nan))) * 100  # Avoid division by zero
    nrmse_ozone = np.sqrt(mse_ozone) / (y_test['Ozone'].max() - y_test['Ozone'].min())

    # Append metrics for PM2.5
    mse_scores_pm25.append(mse_pm25)
    mae_scores_pm25.append(mae_pm25)
    r2_scores_pm25.append(r2_pm25)
    mape_scores_pm25.append(mape_pm25)
    nrmse_scores_pm25.append(nrmse_pm25)

    # Append metrics for Ozone
    mse_scores_ozone.append(mse_ozone)
    mae_scores_ozone.append(mae_ozone)
    r2_scores_ozone.append(r2_ozone)
    mape_scores_ozone.append(mape_ozone)
    nrmse_scores_ozone.append(nrmse_ozone)

# Calculate mean and standard deviation for each metric for PM2.5
mean_mse_pm25 = np.mean(mse_scores_pm25)
std_mse_pm25 = np.std(mse_scores_pm25)
mean_mae_pm25 = np.mean(mae_scores_pm25)
std_mae_pm25 = np.std(mae_scores_pm25)
mean_r2_pm25 = np.mean(r2_scores_pm25)
std_r2_pm25 = np.std(r2_scores_pm25)
mean_mape_pm25 = np.mean(mape_scores_pm25)
std_mape_pm25 = np.std(mape_scores_pm25)
mean_nrmse_pm25 = np.mean(nrmse_scores_pm25)
std_nrmse_pm25 = np.std(nrmse_scores_pm25)

# Calculate mean and standard deviation for each metric for Ozone
mean_mse_ozone = np.mean(mse_scores_ozone)
std_mse_ozone = np.std(mse_scores_ozone)
mean_mae_ozone = np.mean(mae_scores_ozone)
std_mae_ozone = np.std(mae_scores_ozone)
mean_r2_ozone = np.mean(r2_scores_ozone)
std_r2_ozone = np.std(r2_scores_ozone)
mean_mape_ozone = np.mean(mape_scores_ozone)
std_mape_ozone = np.std(mape_scores_ozone)
mean_nrmse_ozone = np.mean(nrmse_scores_ozone)
std_nrmse_ozone = np.std(nrmse_scores_ozone)

# Print evaluation metrics for PM2.5
print(f"PM2.5 - Mean Squared Error: {round(mean_mse_pm25, 2)} ± {round(std_mse_pm25, 2)}")
print(f"PM2.5 - Mean Absolute Error: {round(mean_mae_pm25, 2)} ± {round(std_mae_pm25, 2)}")
print(f"PM2.5 - R^2 Score: {round(mean_r2_pm25, 2)} ± {round(std_r2_pm25, 2)}")
print(f"PM2.5 - Mean Absolute Percentage Error: {round(mean_mape_pm25, 2)}% ± {round(std_mape_pm25, 2)}%")
print(f"PM2.5 - Normalized RMSE: {round(mean_nrmse_pm25, 4)} ± {round(std_nrmse_pm25, 4)}")

# Print evaluation metrics for Ozone
print(f"Ozone - Mean Squared Error: {round(mean_mse_ozone, 2)} ± {round(std_mse_ozone, 2)}")
print(f"Ozone - Mean Absolute Error: {round(mean_mae_ozone, 2)} ± {round(std_mae_ozone, 2)}")
print(f"Ozone - R^2 Score: {round(mean_r2_ozone, 2)} ± {round(std_r2_ozone, 2)}")
print(f"Ozone - Mean Absolute Percentage Error: {round(mean_mape_ozone, 2)}% ± {round(std_mape_ozone, 2)}%")
print(f"Ozone - Normalized RMSE: {round(mean_nrmse_ozone, 4)} ± {round(std_nrmse_ozone, 4)}")


C:\Users\Faculty\AppData\Local\Temp\ipykernel_33400\1998756591.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds


PM2.5 - Mean Squared Error: 786.68 ± 20.91
PM2.5 - Mean Absolute Error: 16.17 ± 0.07
PM2.5 - R^2 Score: 0.9 ± 0.0
PM2.5 - Mean Absolute Percentage Error: 48.25% ± 1.84%
PM2.5 - Normalized RMSE: 0.0281 ± 0.0004
Ozone - Mean Squared Error: 247.11 ± 1.51
Ozone - Mean Absolute Error: 8.98 ± 0.03
Ozone - R^2 Score: 0.79 ± 0.0
Ozone - Mean Absolute Percentage Error: 111.19% ± 1.91%
Ozone - Normalized RMSE: 0.0786 ± 0.0002


In [17]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Input
from tensorflow.keras import models
from tensorflow.keras.utils import plot_model

# Load dataset
data = pd.read_csv(r'C:\Users\Faculty\Desktop\Manoj_Honors\RF_csv\all_available(t-1)_with_spatial.csv')

# Select the relevant columns to create X
X = data[['Time', 'Lat', 'Lon', 'Ozone', 'Average_Ozone_t-1', 
          'NO2', 'Average_NO2_t-1']]

# Target variable
y = data['PM2.5']

# Convert 'Time' to numerical value (timestamp)
X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9

# Normalize the features and target variable using StandardScaler
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))  # y needs to be 2D

# Reshape X to 3D for LSTM input: (samples, timesteps, features)
X_reshaped = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y_scaled, test_size=0.2, random_state=42)

# Define the model with a single input
input_layer = Input(shape=(X_train.shape[1], X_train.shape[2]))  # Shape for the merged input

# LSTM layer with bidirectional LSTM
lstm = Bidirectional(LSTM(128, return_sequences=True))(input_layer)
lstm = Dropout(0.4)(lstm)
lstm = Bidirectional(LSTM(128, return_sequences=True))(input_layer)
lstm = Dropout(0.4)(lstm)
lstm = LSTM(64)(lstm)

# Final dense layers
dense = Dense(64, activation='relu')(lstm)
dense = Dropout(0.4)(dense)
output = Dense(1)(dense)

# Define and compile the model
model = models.Model(inputs=input_layer, outputs=output)
model.compile(optimizer='adam', loss='mse', metrics=['mae', 'mse'])  # Using Adam optimizer

# Summary of the model
model.summary()

# Plot the model architecture
plot_model(model, show_layer_names=True, show_shapes=True)

# Train the model with early stopping
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(
    X_train, y_train, 
    epochs=200, batch_size=64,  # Adjusted epochs and batch size
    validation_data=(X_test, y_test), 
    callbacks=[early_stopping]
)

# Evaluate the model
y_pred = model.predict(X_test)

# Inverse transform predictions and true values to get them back to the original scale
y_pred_rescaled = scaler_y.inverse_transform(y_pred)
y_true_rescaled = scaler_y.inverse_transform(y_test)

# Calculate metrics
mse = mean_squared_error(y_true_rescaled, y_pred_rescaled)
mae = mean_absolute_error(y_true_rescaled, y_pred_rescaled)
rmse = np.sqrt(mse)
r2 = r2_score(y_true_rescaled, y_pred_rescaled)

print(f"MSE: {mse:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")


C:\Users\Faculty\AppData\Local\Temp\ipykernel_154900\2382253863.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 1, 7)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 1, 256)         │       139,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 1, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_18 (LSTM)                  │ (None, 64)             │        82,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,665 (881.50 KB)

 Trainable params: 225,665 (881.50 KB)

 Non-trainable params: 0 (0.00 B)

You must install pydot (`pip install pydot`) for `plot_model` to work.
Epoch 1/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - loss: 0.6077 - mae: 0.5311 - mse: 0.6077 - val_loss: 0.4564 - val_mae: 0.4450 - val_mse: 0.4564
Epoch 2/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - loss: 0.4859 - mae: 0.4627 - mse: 0.4859 - val_loss: 0.4263 - val_mae: 0.4321 - val_mse: 0.4263
Epoch 3/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - loss: 0.4555 - mae: 0.4496 - mse: 0.4555 - val_loss: 0.4099 - val_mae: 0.4163 - val_mse: 0.4099
Epoch 4/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - loss: 0.4418 - mae: 0.4419 - mse: 0.4418 - val_loss: 0.3978 - val_mae: 0.4078 - val_mse: 0.3978
Epoch 5/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - loss: 0.4350 - mae: 0.4371 - mse: 0.4350 - val_loss: 0.3807 - val_mae: 0.4042 - val_mse: 0.3807
Epoch 6/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - loss: 0.4190 - mae: 0.4305 - mse: 0.4190 - val_loss: 0.3752 - val_mae: 0.4039 - val_mse: 

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, RepeatVector, TimeDistributed, Input
from tensorflow.keras import models
from tensorflow.keras.utils import plot_model

# Load dataset
data = pd.read_csv(r'C:\Users\Faculty\Desktop\Manoj_Honors\RF_csv\all_available(t-1)_with_spatial.csv')

# Select the relevant columns to create X
X = data[['Time', 'Lat', 'Lon', 'Ozone', 'Average_Ozone_t-1', 
          'NO2', 'Average_NO2_t-1']]

# Target variable
y = data['PM2.5']

# Convert 'Time' to numerical value (timestamp)
X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9

# Normalize the features and target variable using StandardScaler
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))  # y needs to be 2D

# Reshape X to 3D for LSTM input: (samples, timesteps, features)
X_reshaped = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y_scaled, test_size=0.2, random_state=42)

# Define the Encoder-Decoder model
# Encoder
encoder_input = Input(shape=(X_train.shape[1], X_train.shape[2]))
encoder = Bidirectional(LSTM(128, return_sequences=True))(encoder_input)
encoder = Dropout(0.4)(encoder)
encoder = Bidirectional(LSTM(128))(encoder)
encoder_output = Dense(64, activation='relu')(encoder)

# Decoder
decoder_input = RepeatVector(1)(encoder_output)  # Repeat the context vector for the decoder
decoder = LSTM(64, return_sequences=True)(decoder_input)
decoder = Dropout(0.4)(decoder)
decoder_output = TimeDistributed(Dense(1))(decoder)  # Single time-step prediction

# Define and compile the model
model = models.Model(inputs=encoder_input, outputs=decoder_output)
model.compile(optimizer='adam', loss='mse', metrics=['mae', 'mse'])

# Summary of the model
model.summary()

# Plot the model architecture
plot_model(model, show_layer_names=True, show_shapes=True)

# Train the model with early stopping
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history_Encoder = model.fit(
    X_train, y_train, 
    epochs=200, batch_size=64,  # Adjusted epochs and batch size
    validation_data=(X_test, y_test), 
    callbacks=[early_stopping]
)

# Evaluate the model
y_pred = model.predict(X_test)

# Inverse transform predictions and true values to get them back to the original scale
y_pred_rescaled = scaler_y.inverse_transform(y_pred.reshape(-1, 1))
y_true_rescaled = scaler_y.inverse_transform(y_test)

# Calculate metrics
mse = mean_squared_error(y_true_rescaled, y_pred_rescaled)
mae = mean_absolute_error(y_true_rescaled, y_pred_rescaled)
rmse = np.sqrt(mse)
r2 = r2_score(y_true_rescaled, y_pred_rescaled)

print(f"MSE: {mse:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")

C:\Users\Faculty\AppData\Local\Temp\ipykernel_43276\865289357.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1, 7)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 1, 256)         │       139,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 1, 64)          │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 1, 1)           │            65 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 583,041 (2.22 MB)

 Trainable params: 583,041 (2.22 MB)

 Non-trainable params: 0 (0.00 B)

You must install pydot (`pip install pydot`) for `plot_model` to work.
Epoch 1/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 77s 7ms/step - loss: 0.5835 - mae: 0.5140 - mse: 0.5835 - val_loss: 0.4360 - val_mae: 0.4342 - val_mse: 0.4360
Epoch 2/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 45s 4ms/step - loss: 0.4617 - mae: 0.4502 - mse: 0.4617 - val_loss: 0.4207 - val_mae: 0.4272 - val_mse: 0.4207
Epoch 3/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 87s 8ms/step - loss: 0.4412 - mae: 0.4386 - mse: 0.4412 - val_loss: 0.3969 - val_mae: 0.4107 - val_mse: 0.3969
Epoch 4/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 143s 8ms/step - loss: 0.4150 - mae: 0.4270 - mse: 0.4150 - val_loss: 0.3812 - val_mae: 0.4018 - val_mse: 0.3812
Epoch 5/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 141s 8ms/step - loss: 0.4054 - mae: 0.4204 - mse: 0.4054 - val_loss: 0.3610 - val_mae: 0.3962 - val_mse: 0.3610
Epoch 6/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 87s 8ms/step - loss: 0.3896 - mae: 0.4129 - mse: 0.3896 - val_loss: 0.3523 - val_mae: 0.3885 - val_mse

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense, Dropout, TimeDistributed, Input, Attention, Concatenate
from tensorflow.keras import models
from tensorflow.keras.utils import plot_model

# Load dataset
data = pd.read_csv(r'C:\Users\Faculty\Desktop\Manoj_Honors\RF_csv\all_available(t-1)_with_spatial.csv')

# Select the relevant columns to create X
X = data[['Time', 'Lat', 'Lon', 'Ozone', 'Average_Ozone_t-1', 
          'NO2', 'Average_NO2_t-1']]

# Target variable
y = data['PM2.5']

# Convert 'Time' to numerical value (timestamp)
X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9

# Normalize the features and target variable using StandardScaler
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))  # y needs to be 2D

# Reshape X to 3D for LSTM input: (samples, timesteps, features)
X_reshaped = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y_scaled, test_size=0.2, random_state=42)

# Define the Encoder-Decoder model
# Encoder
encoder_input = Input(shape=(X_train.shape[1], X_train.shape[2]))
encoder = LSTM(128, return_sequences=True)(encoder_input)
encoder = Dropout(0.4)(encoder)
encoder = LSTM(128, return_sequences=True)(encoder)
encoder_output = LSTM(128, return_sequences=True)(encoder)

# Attention Layer
attention = Attention()([encoder_output, encoder_output])  # Self-attention
attention_output = Concatenate()([encoder_output, attention])

# Decoder
decoder = LSTM(64, return_sequences=True)(attention_output)
decoder = Dropout(0.4)(decoder)
decoder_output = TimeDistributed(Dense(1))(decoder)  # Single time-step prediction

# Define and compile the model
model = models.Model(inputs=encoder_input, outputs=decoder_output)
model.compile(optimizer='adam', loss='mse', metrics=['mae', 'mse'])

# Summary of the model
model.summary()

# Plot the model architecture
plot_model(model, show_layer_names=True, show_shapes=True)

# Train the model with early stopping
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history_encoder_attention = model.fit(
    X_train, y_train, 
    epochs=200, batch_size=64,  # Adjusted epochs and batch size
    validation_data=(X_test, y_test), 
    callbacks=[early_stopping]
)

# Evaluate the model
y_pred = model.predict(X_test)

# Inverse transform predictions and true values to get them back to the original scale
y_pred_rescaled = scaler_y.inverse_transform(y_pred.reshape(-1, 1))
y_true_rescaled = scaler_y.inverse_transform(y_test)

# Calculate metrics
mse = mean_squared_error(y_true_rescaled, y_pred_rescaled)
mae = mean_absolute_error(y_true_rescaled, y_pred_rescaled)
rmse = np.sqrt(mse)
r2 = r2_score(y_true_rescaled, y_pred_rescaled)

print(f"MSE: {mse:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")

C:\Users\Faculty\AppData\Local\Temp\ipykernel_43276\3999856011.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 1, 7)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ (None, 1, 128)    │     69,632 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 1, 128)    │          0 │ lstm_7[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_8 (LSTM)       │ (None, 1, 128)    │    131,584 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_9 (LSTM)       │ (None, 1, 128)    │    131,584 │ lstm_8[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_1         │ (None, 1, 128)    │          0 │ lstm_9[0][0],     │
│ (Attention)         │                   │            │ lstm_9[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 1, 256)    │          0 │ lstm_9[0][0],     │
│ (Concatenate)       │                   │            │ attention_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_10 (LSTM)      │ (None, 1, 64)     │     82,176 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 1, 64)     │          0 │ lstm_10[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_2  │ (None, 1, 1)      │         65 │ dropout_5[0][0]   │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 415,041 (1.58 MB)

 Trainable params: 415,041 (1.58 MB)

...

You must install pydot (`pip install pydot`) for `plot_model` to work.
Epoch 1/200


c:\Users\Faculty\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:545: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


10707/10707 ━━━━━━━━━━━━━━━━━━━━ 35s 3ms/step - loss: 0.6685 - mae: 0.5614 - mse: 0.6685 - val_loss: 0.4458 - val_mae: 0.4347 - val_mse: 0.4458
Epoch 2/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 33s 3ms/step - loss: 0.4738 - mae: 0.4559 - mse: 0.4738 - val_loss: 0.4236 - val_mae: 0.4342 - val_mse: 0.4236
Epoch 3/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 33s 3ms/step - loss: 0.4472 - mae: 0.4435 - mse: 0.4472 - val_loss: 0.4063 - val_mae: 0.4166 - val_mse: 0.4063
Epoch 4/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 34s 3ms/step - loss: 0.4319 - mae: 0.4352 - mse: 0.4319 - val_loss: 0.3917 - val_mae: 0.4116 - val_mse: 0.3917
Epoch 5/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 34s 3ms/step - loss: 0.4224 - mae: 0.4299 - mse: 0.4224 - val_loss: 0.3767 - val_mae: 0.3990 - val_mse: 0.3767
Epoch 6/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 34s 3ms/step - loss: 0.4081 - mae: 0.4220 - mse: 0.4081 - val_loss: 0.3679 - val_mae: 0.3954 - val_mse: 0.3679
Epoch 7/200
10707/10707 ━━━━━━━━━━━━━━━━━━━━ 34s 3ms/step - loss: 0.4005 - m

c:\Users\Faculty\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:545: UserWarning: You are using a softmax over axis -1 of a tensor of shape (32, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


5354/5354 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step
MSE: 2217.3769, MAE: 30.8139, RMSE: 47.0890, R2: 0.7239


In [2]:
# Load the dataset
all_available = pd.read_csv(r'D:\Manoj_Honors\RF_csv\all_available(t-1)_with_spatial.csv')

# Prepare the feature set and target variable
X = all_available[['Time', 'Lat','Lon','Ozone','NO2','Average_Ozone_t-1','Average_NO2_t-1']]
y = all_available['PM2.5']

# Convert 'Time' to numeric if it's in datetime format
X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds

# Initialize the Random Forest model
rf = RandomForestRegressor(n_estimators=100, random_state=12)

# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=12)

# Prepare lists to store scores for each fold
mse_scores = []
mae_scores = []
r2_scores = []
mape_scores = []
nrmse_scores = []

# Perform cross-validation
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Fit the model on the training data
    rf.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred = rf.predict(X_test)

    # Calculate evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    nrmse = np.sqrt(mse) / (y_test.max() - y_test.min())

    mse_scores.append(mse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    mape_scores.append(mape)
    nrmse_scores.append(nrmse)

# Calculate mean and standard deviation for each metric
mean_mse = np.mean(mse_scores)
std_mse = np.std(mse_scores)
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)
mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)
mean_mape = np.mean(mape_scores)
std_mape = np.std(mape_scores)
mean_nrmse = np.mean(nrmse_scores)
std_nrmse = np.std(nrmse_scores)

# Print evaluation metrics
print(f"Mean Squared Error: {round(mean_mse, 2)} ± {round(std_mse, 2)}")
print(f"Mean Absolute Error: {round(mean_mae, 2)} ± {round(std_mae, 2)}")
print(f"R^2 Score: {round(mean_r2, 2)} ± {round(std_r2, 2)}")
print(f"Mean Absolute Percentage Error: {round(mean_mape, 2)}% ± {round(std_mape, 2)}%")
print(f"Normalized RMSE: {round(mean_nrmse, 4)} ± {round(std_nrmse, 4)}")

import joblib
model_output_path = r'D:\Manoj_Honors\random_forest_model_PM2.5.joblib'
joblib.dump(rf, model_output_path)

print(f"Model saved to {model_output_path}")




C:\Users\Faculty\AppData\Local\Temp\ipykernel_20740\3618955249.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Time'] = pd.to_datetime(X['Time'], errors='coerce').astype('int64') // 10**9  # Convert to seconds


Mean Squared Error: 747.29 ± 22.14
Mean Absolute Error: 15.51 ± 0.06
R^2 Score: 0.91 ± 0.0
Mean Absolute Percentage Error: 46.88% ± 2.27%
Normalized RMSE: 0.0274 ± 0.0004
Model saved to D:\Manoj_Honors\random_forest_model_PM2.5.joblib


In [2]:
import pandas as pd
import joblib

# File paths
model_path = r"D:\Manoj_Honors\random_forest_model_PM2.5.joblib"
input_csv_path = r'D:\Manoj_Honors\Imputed_Predictions(NO2,Ozone).csv'
output_csv_path = r"D:\Manoj_Honors\Imputed_Predictions(Univariate).csv"

# Load the model
with open(model_path, 'rb') as model_file:
    rf_model = joblib.load(model_file)

# Load the input data
df = pd.read_csv(input_csv_path)

# Check the required columns
required_columns = ['Time', 'Lat','Lon','NO2','PM2.5','Average_NO2_t-1','Average_Ozone_t-1','Ozone']
for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# Identify rows where both PM2.5 and NO2 are missing, but Ozone is available
mask_missing_pm_no2 = df['NO2'].notna() & df['Ozone'].notna() & df['PM2.5'].isna()

# Count such rows
missing_rows_count = mask_missing_pm_no2.sum()
print(f"Number of rows with missing PM2.5 :  {missing_rows_count}")

# Filter the rows that meet the condition
rows_to_impute = df[mask_missing_pm_no2]

# Prepare the features for imputation (Time, Lat, Lon, Ozone, Average_Ozone_t-1, etc.)
X_to_impute = rows_to_impute[['Time', 'Lat','Lon','Ozone','NO2','Average_Ozone_t-1','Average_NO2_t-1']]

# Convert 'Time' to seconds since epoch for prediction input
X_to_impute.loc[:, 'Time'] = pd.to_datetime(X_to_impute['Time'], errors='coerce').astype('int64') // 10**9

# Ensure the columns are in the same order as during model training
train_columns = ['Time', 'Lat','Lon','Ozone','NO2','Average_Ozone_t-1','Average_NO2_t-1']
X_to_impute = X_to_impute[train_columns]

# Impute the missing values using the model
imputed_values = rf_model.predict(X_to_impute)

# Assign the imputed values to the 'NO2' column in the original DataFrame
df.loc[mask_missing_pm_no2, 'PM2.5'] = imputed_values

# Save the updated dataframe with imputed values to a new CSV
df.to_csv(output_csv_path, index=False)

print(f"Imputed rows and saved the updated data to {output_csv_path}")


Number of rows with missing PM2.5 :  32018
Imputed rows and saved the updated data to D:\Manoj_Honors\Imputed_Predictions(Univariate).csv
